# Temporally coherent BAPPS-style video augmentations

This notebook previews distortions for a video perceptual-similarity dataset. The reusable implementation lives in the importable wan_lpips.video_augmentations package module; this file only configures and runs the experiment.

The [BAPPS/LPIPS paper](https://openaccess.thecvf.com/content_cvpr_2018/papers/Zhang_The_Unreasonable_Effectiveness_CVPR_2018_paper.pdf) builds traditional examples from photometric, corruption, noise, blur, spatial, and compression atoms, including sequential compositions. The video analogue here keeps each atom's parameters fixed or smoothly varying through time, avoiding independently randomized frame transforms that create accidental flicker.

For every clip in video_samples, all registered variants are generated:

1. **Photometric drift:** smooth exposure, contrast, and saturation trajectories.
2. **Color:** coherent temperature, tint, gamma, color removal, and vignetting.
3. **Spatial:** camera drift, nonlinear wave warp, and chromatic aberration.
4. **Blur/resample:** the original temporally fixed low-pass/downsample degradation.
5. **Temporal:** stronger causal ghosting with recent frames.
6. **Local/flicker:** moving regional exposure flicker, local elastic warp, and bursty displaced blocks.
7. **Corruption:** stronger AR(1) noise/quantization and a stable checkerboard artifact.
8. **Compression:** one JPEG quality held constant over the clip.
9. **Composition:** a BAPPS-style chain of photometric, chromatic, and noise atoms.

Each transform accepts severity in [0,1], is deterministic for a sample/name/seed, and preserves resolution, frame count, and [0,1] range. Runtime dependencies are torch, NumPy, Pillow, IPython, and PyAV.


In [ ]:
from wan_lpips.video_augmentations import (
    AUGMENTATIONS,
    AugmentationConfig,
    PACKAGE_ROOT,
    discover_sample_paths,
    render_sample,
    run_synthetic_contract_tests,
)


In [ ]:
SAMPLE_DIR = PACKAGE_ROOT / 'wan-lpips' / 'video_samples'

cfg = AugmentationConfig(
    preview_max_edge=512,  # Set to None for full-resolution export.
    severity=0.65,
    seed=2026,
    encode_crf=25,
    save_outputs=False,
    output_dir=PACKAGE_ROOT / 'wan-lpips' / 'video_augmentation_outputs',
)

sample_paths = discover_sample_paths(SAMPLE_DIR)
print(f'device: {cfg.device}')
print(f'augmentations: {list(AUGMENTATIONS)}')
print(f'samples: {[path.name for path in sample_paths]}')


In [ ]:
run_synthetic_contract_tests(cfg.device, cfg.seed)
print(f'All {len(AUGMENTATIONS)} augmentation contracts passed.')


In [ ]:
all_diagnostics = {}
for sample_path in sample_paths:
    all_diagnostics[sample_path.name] = render_sample(
        sample_path,
        cfg,
        display_inline=True,
    )


## Turning this into a dataset generator

- Draw severity from a recorded distribution and store augmentation name, severity, and deterministic seed beside every pair.
- Keep timestamps, crop, frame count, and resolution identical unless temporal misalignment is the distortion being studied.
- Split by source video before generating variants so one source cannot cross train/validation boundaries.
- Mix these traditional distortions with failures from real restoration, interpolation, compression, and generation algorithms, mirroring BAPPS's synthetic/algorithmic mixture.
- Retain repeated human vote counts rather than only scalar averages.
- Inspect high residual-Δt diagnostics for accidental flicker before large-scale export.
